In [6]:
import gemmi
import numpy as np
from scipy import stats, ndimage  # Added ndimage
import sys

def _compute_metrics(data1, data2):
    """Helper to calculate Pearson CC and Spearman for a given set of data."""
    if data1.size == 0 or data2.size == 0:
        return 0.0, 0.0, 0.0

    # Pearson Correlation
    mean1 = np.mean(data1, dtype=np.float64)
    mean2 = np.mean(data2, dtype=np.float64)

    c1 = data1 - mean1
    c2 = data2 - mean2

    numerator = np.dot(c1, c2)
    denominator = np.linalg.norm(c1) * np.linalg.norm(c2)

    pearson_cc = 0.0
    if denominator != 0:
        pearson_cc = numerator / denominator

    # Spearman Correlation
    res_spearman = stats.spearmanr(data1, data2)

    return pearson_cc, res_spearman.statistic, res_spearman.pvalue

def calculate_split_map_cc(map_path_1, map_path_2, model_path):
    try:
        # --- 1. READ MAPS WITH GEMMI ---
        # setup(0.0) replaces NaNs with 0.0 to prevent errors
        m1 = gemmi.read_ccp4_map(map_path_1)
        m1.setup(0.0)
        grid1 = m1.grid

        m2 = gemmi.read_ccp4_map(map_path_2)
        m2.setup(0.0)
        grid2 = m2.grid

        # --- 2. MATCH GRID SHAPES (ROBUST RESAMPLING) ---
        arr1 = np.array(grid1, copy=False)
        arr2_raw = np.array(grid2, copy=False)

        # Check if dimensions match (nu, nv, nw)
        if arr1.shape != arr2_raw.shape:
            print(f"Notice: Grid shape mismatch {arr1.shape} vs {arr2_raw.shape}. Resampling Map 2...", file=sys.stderr)

            # We calculate the scale factors to map indices from Grid 1 to Grid 2
            # Shape is (nu, nv, nw)
            factors = [
                arr2_raw.shape[0] / arr1.shape[0], # u scale
                arr2_raw.shape[1] / arr1.shape[1], # v scale
                arr2_raw.shape[2] / arr1.shape[2]  # w scale
            ]

            # Create a coordinate grid for the target shape (Map 1)
            # This generates the indices (0,0,0), (0,0,1)... for the new grid
            coords = np.mgrid[0:arr1.shape[0], 0:arr1.shape[1], 0:arr1.shape[2]]

            # Scale these coordinates to match the source grid (Map 2)
            coords[0] = coords[0] * factors[0]
            coords[1] = coords[1] * factors[1]
            coords[2] = coords[2] * factors[2]

            # Interpolate Map 2 at these coordinates
            # mode='wrap' handles the periodic boundaries of the map (crystal lattice)
            # order=1 is linear interpolation (standard), use 3 for cubic
            arr2 = ndimage.map_coordinates(arr2_raw, coords, order=1, mode='wrap')

        else:
            arr2 = arr2_raw

        # --- 3. GENERATE SOLVENT MASK ---
        structure = gemmi.read_structure(model_path)

        # Create a grid for the mask that matches Map 1 exactly
        mask_grid = grid1.clone()
        mask_grid.fill(0)

        masker = gemmi.SolventMasker(gemmi.AtomicRadiiSet.Cctbx)
        # 0 = Protein/Structure, 1 = Solvent
        masker.put_mask_on_float_grid(mask_grid, structure[0])

        mask_array = np.array(mask_grid, copy=False)

        if arr1.size != mask_array.size:
             print(f"Error: Critical size mismatch. Map: {arr1.size}, Mask: {mask_array.size}", file=sys.stderr)
             return None

        # --- 4. PREPARE DATA ---
        flat1 = arr1.ravel()
        flat2 = arr2.ravel()
        flat_mask = mask_array.ravel()

        results = {}

        # --- 5. CALCULATE OVERALL CC ---
        o_cc, o_sp, o_pval = _compute_metrics(flat1, flat2)
        results['overall'] = {'cc': o_cc, 'spearman': o_sp, 'n': flat1.size}

        # --- 6. SPLIT CALCULATION ---
        # Gemmi SolventMasker: 1.0 is solvent, 0.0 is protein
        is_solvent = flat_mask > 0.5
        is_protein = ~is_solvent

        # Protein Region
        p_data1 = flat1[is_protein]
        p_data2 = flat2[is_protein]

        if p_data1.size > 0:
            p_cc, p_sp, p_pval = _compute_metrics(p_data1, p_data2)
            results['protein'] = {'cc': p_cc, 'spearman': p_sp, 'n': p_data1.size}
        else:
            results['protein'] = {'cc': 0, 'spearman': 0, 'n': 0}

        # Solvent Region
        s_data1 = flat1[is_solvent]
        s_data2 = flat2[is_solvent]

        if s_data1.size > 0:
            s_cc, s_sp, s_pval = _compute_metrics(s_data1, s_data2)
            results['solvent'] = {'cc': s_cc, 'spearman': s_sp, 'n': s_data1.size}
        else:
            results['solvent'] = {'cc': 0, 'spearman': 0, 'n': 0}

        return results

    except Exception as e:
        print(f"Error in Split CC calc: {e}", file=sys.stderr)
        import traceback
        traceback.print_exc()
        return None

In [7]:
id="2QE6"
model_path = f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/exp_1/debias_validation_atoms_01/{id}/processed/{id}_updated.pdb"
calculate_split_map_cc(f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/merged_intensities/{id}-sf/baseline_maps/{id.lower()}_validation_2fo-fc_map_coef.ccp4",f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/exp_1/debias_validation_atoms_01/{id}/quantify_results/k_1.0_cap_99/{id}_mean.ccp4", model_path)

Notice: Grid shape mismatch (108, 80, 90) vs (160, 120, 128). Resampling Map 2...


{'overall': {'cc': 0.7482271, 'spearman': 0.6607409067906214, 'n': 777600},
 'protein': {'cc': 0.78309715, 'spearman': 0.7345254634146275, 'n': 422748},
 'solvent': {'cc': 0.54228795, 'spearman': 0.42310431716542557, 'n': 354852}}

In [8]:
calculate_split_map_cc(f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/merged_intensities/{id}-sf/baseline_maps/{id.lower()}_validation_2fo-fc_map_coef.ccp4",f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/exp_1/debias_validation_atoms_01/{id}/quantify_results/k_0.2_cap_99/{id}_mean.ccp4", model_path)

Notice: Grid shape mismatch (108, 80, 90) vs (160, 120, 128). Resampling Map 2...


{'overall': {'cc': 0.74824524, 'spearman': 0.6607444438698914, 'n': 777600},
 'protein': {'cc': 0.78311574, 'spearman': 0.7345375656098557, 'n': 422748},
 'solvent': {'cc': 0.54228795, 'spearman': 0.42310431716542557, 'n': 354852}}

In [9]:
calculate_split_map_cc(f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/merged_intensities/{id}-sf/baseline_maps/calculated_density.ccp4",f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/merged_intensities/{id}-sf/baseline_maps/{id.lower()}_validation_2fo-fc_map_coef.ccp4", model_path)

Notice: Grid shape mismatch (160, 120, 128) vs (108, 80, 90). Resampling Map 2...


{'overall': {'cc': 0.39197686, 'spearman': 0.26920733834024463, 'n': 2457600},
 'protein': {'cc': 0.46977192, 'spearman': 0.434651336653513, 'n': 1094300},
 'solvent': {'cc': 0.16585147, 'spearman': 0.026899474582494615, 'n': 1363300}}

In [10]:
calculate_split_map_cc(f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/merged_intensities/{id}-sf/baseline_maps/calculated_density.ccp4",f"/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/exp_1/debias_validation_atoms_01/{id}/quantify_results/k_1.0_cap_99/{id}_mean.ccp4", model_path)

{'overall': {'cc': 0.47967786, 'spearman': 0.30373261247426037, 'n': 2457600},
 'protein': {'cc': 0.61844516, 'spearman': 0.5554135933933461, 'n': 1094300},
 'solvent': {'cc': 0.07764724, 'spearman': -0.05943350668631641, 'n': 1363300}}

In [15]:
import gemmi
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats, ndimage
import sys
import os

# ==========================================
# 1. ROBUST METRIC CALCULATOR
# ==========================================

def calculate_split_map_cc(map_path_1, map_path_2, model_path):
    """
    Calculates CC, Spearman, and Log-Likelihood for Overall, Protein, and Solvent regions.
    Robust to grid shape mismatches via interpolation.
    """
    try:
        # --- Read Maps ---
        m1 = gemmi.read_ccp4_map(map_path_1)
        m1.setup(0.0) # NaNs -> 0.0
        grid1 = m1.grid

        m2 = gemmi.read_ccp4_map(map_path_2)
        m2.setup(0.0)
        grid2 = m2.grid

        arr1 = np.array(grid1, copy=False)
        arr2_raw = np.array(grid2, copy=False)

        # --- Grid Matching (Interpolation) ---
        # If dimensions differ, interpolate Map 2 onto Map 1's grid
        if arr1.shape != arr2_raw.shape:
            # Scale factors
            factors = [
                arr2_raw.shape[0] / arr1.shape[0],
                arr2_raw.shape[1] / arr1.shape[1],
                arr2_raw.shape[2] / arr1.shape[2]
            ]
            # Coordinate grid
            coords = np.mgrid[0:arr1.shape[0], 0:arr1.shape[1], 0:arr1.shape[2]]
            coords[0] = coords[0] * factors[0]
            coords[1] = coords[1] * factors[1]
            coords[2] = coords[2] * factors[2]

            # Interpolate (Linear order=1 is faster and sufficient for CC)
            arr2 = ndimage.map_coordinates(arr2_raw, coords, order=1, mode='wrap')
        else:
            arr2 = arr2_raw

        # --- Solvent Mask ---
        structure = gemmi.read_structure(model_path)
        mask_grid = grid1.clone()
        mask_grid.fill(0)
        masker = gemmi.SolventMasker(gemmi.AtomicRadiiSet.Cctbx)
        masker.put_mask_on_float_grid(mask_grid, structure[0])
        mask_array = np.array(mask_grid, copy=False)

        if arr1.size != mask_array.size:
             return None

        # --- Helper for Metrics ---
        def get_metrics(d1, d2):
            if d1.size == 0: return 0.0, 0.0, 0.0, 0.0

            # Pearson
            c1 = d1 - np.mean(d1)
            c2 = d2 - np.mean(d2)
            num = np.dot(c1, c2)
            den = np.linalg.norm(c1) * np.linalg.norm(c2)
            cc = num / den if den != 0 else 0.0

            # Spearman
            sp = stats.spearmanr(d1, d2).statistic

            # Log Likelihood (Gaussian approximation)
            # LL is proportional to -N * log(MSE)
            residuals = d1 - d2
            mse = np.mean(residuals**2)
            ll = -0.5 * d1.size * np.log(mse) if mse > 1e-12 else float('inf')

            return cc, sp, ll

        flat1 = arr1.ravel()
        flat2 = arr2.ravel()
        flat_mask = mask_array.ravel()

        results = {}

        # 1. Overall
        cc, sp, ll = get_metrics(flat1, flat2)
        results['overall'] = {'cc': cc, 'spearman': sp, 'll': ll}

        # Split Masks (0=Protein, 1=Solvent)
        is_solvent = flat_mask > 0.5
        is_protein = ~is_solvent

        # 2. Protein
        cc, sp, ll = get_metrics(flat1[is_protein], flat2[is_protein])
        results['protein'] = {'cc': cc, 'spearman': sp, 'll': ll}

        # 3. Solvent
        cc, sp, ll = get_metrics(flat1[is_solvent], flat2[is_solvent])
        results['solvent'] = {'cc': cc, 'spearman': sp, 'll': ll}

        return results

    except Exception as e:
        print(f"Error processing maps: {e}", file=sys.stderr)
        return None

# ==========================================
# 2. CHECKPOINTED BATCH PROCESSOR
# ==========================================

def run_batch_analysis(csv_file, output_csv="analysis_report.csv"):
    """
    Reads IDs and resolutions from CSV. Checks output_csv for existing results
    and only computes new ones (checkpointing).
    """
    # Load Input List
    df_input = pd.read_csv(csv_file)

    # Load Existing Results (Checkpoint)
    if os.path.exists(output_csv):
        print(f"Found existing checkpoint file: {output_csv}")
        df_existing = pd.read_csv(output_csv)
        processed_ids = set(df_existing['id'].astype(str))
        final_data = df_existing.to_dict('records')
    else:
        print("No checkpoint found. Starting fresh.")
        processed_ids = set()
        final_data = []

    # Filter input for only unprocessed IDs
    # Note: If you have duplicate IDs in input, this logic handles unique IDs.
    to_process = df_input[~df_input['id'].astype(str).isin(processed_ids)]

    if len(to_process) == 0:
        print("All IDs already processed.")
        return pd.DataFrame(final_data)

    print(f"Processing {len(to_process)} new entries...")

    # Define Base Path
    base_dir = "/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond"

    # Iterate
    for index, row in to_process.iterrows():
        pdb_id = str(row['id'])
        res = float(row['resolution'])
        pdb_lower = pdb_id.lower()

        print(f"Processing {pdb_id}...")

        # Construct Paths
        model_path = f"{base_dir}/exp_1/debias_validation_atoms_01/{pdb_id}/processed/{pdb_id}_updated.pdb"

        # Maps
        target_map = f"{base_dir}/merged_intensities/{pdb_id}-sf/baseline_maps/{pdb_lower}_validation_2fo-fc_map_coef.ccp4"
        debiased_map = f"{base_dir}/exp_1/debias_validation_atoms_01/{pdb_id}/quantify_results/k_1.0_cap_99/{pdb_id}_mean.ccp4"
        calc_map = f"{base_dir}/merged_intensities/{pdb_id}-sf/baseline_maps/calculated_density.ccp4"

        # --- 1. De-biased (Exp) vs Target ---
        if os.path.exists(target_map) and os.path.exists(debiased_map) and os.path.exists(model_path):
            res_debias = calculate_split_map_cc(target_map, debiased_map, model_path)
            if res_debias:
                entry = {
                    'id': pdb_id, 'resolution': res, 'type': 'De-biased (Exp)',
                    'overall_cc': res_debias['overall']['cc'], 'overall_ll': res_debias['overall']['ll'],
                    'protein_cc': res_debias['protein']['cc'], 'protein_ll': res_debias['protein']['ll'],
                    'solvent_cc': res_debias['solvent']['cc'], 'solvent_ll': res_debias['solvent']['ll']
                }
                final_data.append(entry)

        # --- 2. Baseline (Calc) vs Target ---
        if os.path.exists(target_map) and os.path.exists(calc_map) and os.path.exists(model_path):
            res_base = calculate_split_map_cc(target_map, calc_map, model_path)
            if res_base:
                entry = {
                    'id': pdb_id, 'resolution': res, 'type': 'Baseline (Calc)',
                    'overall_cc': res_base['overall']['cc'], 'overall_ll': res_base['overall']['ll'],
                    'protein_cc': res_base['protein']['cc'], 'protein_ll': res_base['protein']['ll'],
                    'solvent_cc': res_base['solvent']['cc'], 'solvent_ll': res_base['solvent']['ll']
                }
                final_data.append(entry)

        # Update checkpoint file after every ID (safer)
        pd.DataFrame(final_data).to_csv(output_csv, index=False)

    return pd.DataFrame(final_data)

# ==========================================
# 3. DETAILED PLOTTING
# ==========================================

def create_single_plot(df, metric_col, title, ylabel, filename, explanation):
    """
    Generic function to create a sorted, clean scatter/line plot.
    """
    plt.figure(figsize=(12, 6))

    # Sort DataFrame by resolution (Ascending: High res -> Low res)
    df_sorted = df.sort_values(by='resolution')

    # Create Labels: "ID (1.5A)"
    df_sorted['label'] = df_sorted['id'] + " (" + df_sorted['resolution'].astype(str) + "Å)"

    # Plot
    sns.scatterplot(
        data=df_sorted,
        x='label',
        y=metric_col,
        hue='type',
        style='type',
        s=100,
        palette={"De-biased (Exp)": "#3498db", "Baseline (Calc)": "#e74c3c"}
    )

    # Style
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Structure ID (Resolution)', fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Map Source', loc='best')

    # Remove grid
    plt.grid(False)
    sns.despine() # Remove top and right borders

    # Save
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close() # Close to free memory

    # Print Explanation
    print(f"\n--- {title} ---")
    print(f"Saved to: {filename}")
    print(explanation)


def plot_comparisons(df):
    """Generates separate plots for each metric."""
    sns.set_theme(style="white", rc={"axes.grid": False}) # Set global style to white background, no grid

    # 1. Overall CC
    create_single_plot(
        df, 'overall_cc', 'Overall Correlation Coefficient', 'CC', 'plot_overall_cc.png',
        "Explanation: Measures how well the density shapes match globally. A value of 1.0 is a perfect match. "
        "This includes both the protein and the bulk solvent regions."
    )

    # 2. Overall LL
    create_single_plot(
        df, 'overall_ll', 'Overall Log-Likelihood', 'Log-Likelihood', 'plot_overall_ll.png',
        "Explanation: Measures how likely the experimental map is given the target, assuming Gaussian errors. "
        "Higher (less negative) values are better. Unlike CC, this is sensitive to the absolute scale/amplitude of the map density."
    )

    # 3. Protein CC
    create_single_plot(
        df, 'protein_cc', 'Protein Region Correlation', 'CC (Protein)', 'plot_protein_cc.png',
        "Explanation: Correlation calculated ONLY within the molecular mask (where atoms exist). "
        "This indicates how well the atomic features (side chains, backbone) are resolved compared to the validation map."
    )

    # 4. Solvent CC
    create_single_plot(
        df, 'solvent_cc', 'Solvent Region Correlation', 'CC (Solvent)', 'plot_solvent_cc.png',
        "Explanation: Correlation calculated ONLY in the solvent region (outside the molecule). "
        "High correlation here suggests the bulk solvent modeling or noise levels match the validation target well."
    )

    # 5. Protein vs Solvent Comparison
    plt.figure(figsize=(10, 8))

    # Create direct comparison
    sns.scatterplot(
        data=df,
        x='protein_cc',
        y='solvent_cc',
        hue='type',
        style='type',
        s=120,
        palette={"De-biased (Exp)": "#3498db", "Baseline (Calc)": "#e74c3c"}
    )

    # Diagonal line
    lims = [min(df[['protein_cc', 'solvent_cc']].min().min(), 0), 1.0]
    plt.plot(lims, lims, '--', color='gray', alpha=0.5, label='Equal Performance')

    plt.title('Protein vs. Solvent Correlation Trade-off', fontsize=16, fontweight='bold')
    plt.xlabel('Protein Correlation', fontsize=12)
    plt.ylabel('Solvent Correlation', fontsize=12)
    plt.legend()
    plt.grid(False)
    sns.despine()

    plt.tight_layout()
    plt.savefig("plot_protein_vs_solvent.png", dpi=300)
    plt.close()

    print("\n--- Protein vs Solvent Trade-off ---")
    print("Saved to: plot_protein_vs_solvent.png")
    print("Explanation: X-axis is Protein Quality, Y-axis is Solvent Quality. Points below the diagonal mean the Protein is resolved better than the Solvent (typical).")


# ==========================================
# MAIN
# ==========================================

if __name__ == "__main__":
    # Your input CSV file
    csv_file_path = "/Users/hamletkhachatryan/Desktop/DPhil_projects/pseudo/pseudo_paper/merged_intensities/diamond/pdbs.csv"

    if os.path.exists(csv_file_path):
        # Run analysis (with checkpoints)
        results_df = run_batch_analysis(csv_file_path)

        # Plot if we have data
        if not results_df.empty:
            plot_comparisons(results_df)
    else:
        print(f"Error: CSV file '{csv_file_path}' not found.")

Found existing checkpoint file: analysis_report.csv
All IDs already processed.

--- Overall Correlation Coefficient ---
Saved to: plot_overall_cc.png
Explanation: Measures how well the density shapes match globally. A value of 1.0 is a perfect match. This includes both the protein and the bulk solvent regions.

--- Overall Log-Likelihood ---
Saved to: plot_overall_ll.png
Explanation: Measures how likely the experimental map is given the target, assuming Gaussian errors. Higher (less negative) values are better. Unlike CC, this is sensitive to the absolute scale/amplitude of the map density.

--- Protein Region Correlation ---
Saved to: plot_protein_cc.png
Explanation: Correlation calculated ONLY within the molecular mask (where atoms exist). This indicates how well the atomic features (side chains, backbone) are resolved compared to the validation map.

--- Solvent Region Correlation ---
Saved to: plot_solvent_cc.png
Explanation: Correlation calculated ONLY in the solvent region (outsi